<a href="https://colab.research.google.com/github/Gabrixl1/MBA-DATA-SCIENCE/blob/main/14DTSR%20/%20reinforcement%20learning%20/%20paulo%20caixeta%20de%20oliveira.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded = files.upload()  # Selecione os 3 CSVs: BRFS3, PETR4, VALE3

Saving BRFS3_Dados_Históricos.csv to BRFS3_Dados_Históricos (1).csv
Saving PETR4_Dados_Históricos.csv to PETR4_Dados_Históricos (1).csv
Saving VALE3_Dados_Históricos.csv to VALE3_Dados_Históricos (1).csv


In [3]:
import numpy, pandas, csv, io, re, warnings, random
from collections import defaultdict

OK


In [4]:
import csv, io, re, warnings
import numpy as np
import pandas as pd
import random
from collections import defaultdict
warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════
# 1. CARGA E LIMPEZA DOS DADOS REAIS
# ══════════════════════════════════════════════════

def load_investing_csv(path, name):
    """Lê CSV exportado do Investing.com (suporta os dois formatos de encoding)."""
    with open(path, "rb") as f:
        raw = f.read().decode("utf-8-sig")

    rows = []
    reader = csv.reader(io.StringIO(raw))
    for row in reader:
        if not row:
            continue
        if len(row) == 1:          # PETR4/VALE3: linha inteira em uma célula
            sub = list(csv.reader([row[0]]))[0]
            rows.append(sub)
        else:
            rows.append(row)       # BRFS3: formato padrão

    df = pd.DataFrame(rows[1:], columns=["Data","Ultimo","Abertura","Maxima","Minima","Vol","Var"])

    df["Data"] = pd.to_datetime(df["Data"], format="%d.%m.%Y", errors="coerce")
    df = df.dropna(subset=["Data"])

    for col in ["Ultimo","Abertura","Maxima","Minima"]:
        df[col] = (df[col].str.replace(".", "", regex=False)
                          .str.replace(",", ".", regex=False)
                          .astype(float))

    def parse_vol(v):
        v = str(v).strip().replace(".", "").replace(",", ".")
        if v.endswith("M"): return float(v[:-1]) * 1e6
        if v.endswith("B"): return float(v[:-1]) * 1e9
        if v.endswith("K"): return float(v[:-1]) * 1e3
        try: return float(v)
        except: return float("nan")

    df["Vol"]  = df["Vol"].apply(parse_vol)
    df["Var"]  = (df["Var"].str.replace("%","",regex=False)
                           .str.replace(",",".",regex=False)
                           .astype(float))
    df = df.sort_values("Data").reset_index(drop=True)
    df.attrs["name"] = name
    return df


# ══════════════════════════════════════════════════
# 2. INDICADORES TÉCNICOS
# ══════════════════════════════════════════════════

def add_indicators(df):
    """Adiciona RSI(14), Momentum(5d), MA_ratio(20d) ao DataFrame."""
    p = df["Ultimo"].values
    n = len(p)

    # RSI (14)
    rsi = np.full(n, 50.0)
    for i in range(14, n):
        diff  = np.diff(p[i-14:i+1])
        gain  = diff[diff > 0].mean() if (diff > 0).any() else 1e-9
        loss  = -diff[diff < 0].mean() if (diff < 0).any() else 1e-9
        rsi[i] = 100 - 100 / (1 + gain / loss)

    # Momentum 5d
    mom = np.full(n, 0.0)
    for i in range(5, n):
        mom[i] = p[i] / p[i-5] - 1

    # Distância da MA20
    ma_ratio = np.full(n, 0.0)
    for i in range(20, n):
        ma_ratio[i] = p[i] / p[i-20:i].mean() - 1

    df["RSI"]      = rsi
    df["Mom"]      = mom
    df["MA_ratio"] = ma_ratio
    return df


# ══════════════════════════════════════════════════
# 3. AMBIENTE DE TRADING
# ══════════════════════════════════════════════════

class TradingEnv:
    """
    Estado  : (RSI_bin, Mom_bin, MA_bin, posição)  → 54 combinações
    Ações   : 0=Vender  1=Manter  2=Comprar
    Recompensa: retorno percentual do portfólio com custo de 0,1%
    """
    TRANSACTION_COST = 0.001

    def __init__(self, df: pd.DataFrame, start_idx=30, initial_cash=10_000.0):
        self.df           = df.reset_index(drop=True)
        self.prices       = df["Ultimo"].values
        self.rsi          = df["RSI"].values
        self.mom          = df["Mom"].values
        self.ma_ratio     = df["MA_ratio"].values
        self.n            = len(df)
        self.start_idx    = start_idx
        self.initial_cash = initial_cash
        self.reset()

    def _bin(self, rsi, mom, mar):
        rsi_b = 0 if rsi < 35 else (2 if rsi > 65 else 1)
        mom_b = 0 if mom < -0.02 else (2 if mom > 0.02 else 1)
        mar_b = 0 if mar < -0.02 else (2 if mar > 0.02 else 1)
        return (rsi_b, mom_b, mar_b)

    def reset(self):
        self.idx      = self.start_idx
        self.position = 0
        self.cash     = self.initial_cash
        self.shares   = 0.0
        self.history  = [self.initial_cash]
        return self._state()

    def _state(self):
        b = self._bin(self.rsi[self.idx], self.mom[self.idx], self.ma_ratio[self.idx])
        return (*b, self.position)

    def _portfolio(self):
        return self.cash + self.shares * self.prices[self.idx]

    def step(self, action):
        price = self.prices[self.idx]

        if action == 2 and self.position == 0:    # Comprar
            self.shares   = self.cash / price * (1 - self.TRANSACTION_COST)
            self.cash     = 0.0
            self.position = 1

        elif action == 0 and self.position == 1:  # Vender
            self.cash     = self.shares * price * (1 - self.TRANSACTION_COST)
            self.shares   = 0.0
            self.position = 0

        self.idx += 1
        done      = self.idx >= self.n - 1
        prev      = self.history[-1]
        curr      = self._portfolio()
        reward    = (curr - prev) / prev
        self.history.append(curr)

        next_state = self._state() if not done else None
        return next_state, reward, done, {"portfolio": curr}


# ══════════════════════════════════════════════════
# 4. AGENTE Q-LEARNING
# ══════════════════════════════════════════════════

class QLearningAgent:
    def __init__(self, alpha=0.1, gamma=0.99, eps=1.0, eps_min=0.05, eps_decay=0.995):
        self.alpha     = alpha
        self.gamma     = gamma
        self.eps       = eps
        self.eps_min   = eps_min
        self.eps_decay = eps_decay
        self.Q         = defaultdict(lambda: np.zeros(3))

    def act(self, state):
        if random.random() < self.eps:
            return random.randint(0, 2)
        return int(np.argmax(self.Q[state]))

    def update(self, s, a, r, s2):
        target         = r if s2 is None else r + self.gamma * np.max(self.Q[s2])
        self.Q[s][a]  += self.alpha * (target - self.Q[s][a])

    def decay(self):
        self.eps = max(self.eps_min, self.eps * self.eps_decay)

    def reset_exploration(self):
        self.eps = 1.0


# ══════════════════════════════════════════════════
# 5. TREINO + AVALIAÇÃO
# ══════════════════════════════════════════════════

def train(df, n_episodes=300, train_frac=0.7, verbose=True, name=""):
    """
    Divide os dados em treino/teste (walk-forward).
    Treina o agente na janela de treino e avalia na de teste.
    """
    split    = int(len(df) * train_frac)
    df_train = df.iloc[:split].reset_index(drop=True)
    df_test  = df.iloc[split:].reset_index(drop=True)

    env   = TradingEnv(df_train)
    agent = QLearningAgent()

    ep_values = []
    for ep in range(1, n_episodes + 1):
        state = env.reset()
        while True:
            action          = agent.act(state)
            next_s, r, done, info = env.step(action)
            agent.update(state, action, r, next_s)
            state = next_s
            if done:
                break
        agent.decay()
        ep_values.append(info["portfolio"])

        if verbose and ep % 75 == 0:
            print(f"  [{name}] ep {ep:>3}/{n_episodes} | ε={agent.eps:.3f} | portfólio treino: R${info['portfolio']:,.0f}")

    return agent, df_train, df_test


def evaluate(agent, df_eval):
    """Avaliação greedy (sem exploração)."""
    env      = TradingEnv(df_eval)
    state    = env.reset()
    saved    = agent.eps
    agent.eps = 0.0

    trades = []
    while True:
        a = agent.act(state)
        ns, r, done, info = env.step(a)
        trades.append({"action": a, "portfolio": info["portfolio"],
                        "price": df_eval["Ultimo"].iloc[env.idx - 1]})
        state = ns
        if done:
            break

    agent.eps = saved
    return pd.DataFrame(trades), np.array(env.history)


def buy_and_hold(df_eval, cash=10_000.0):
    prices = df_eval["Ultimo"].values
    shares = cash / prices[30]
    return np.array([shares * p for p in prices[30:]])


# ══════════════════════════════════════════════════
# 6. MÉTRICAS FINANCEIRAS
# ══════════════════════════════════════════════════

def metrics(portfolio, label=""):
    pv  = np.array(portfolio)
    ret = np.diff(pv) / pv[:-1]
    n   = len(ret)

    total_ret  = (pv[-1] / pv[0] - 1) * 100
    ann_ret    = ((1 + total_ret / 100) ** (252 / n) - 1) * 100
    ann_vol    = ret.std() * np.sqrt(252) * 100
    sharpe     = (ret.mean() * 252) / (ret.std() * np.sqrt(252) + 1e-9)
    running    = np.maximum.accumulate(pv)
    max_dd     = ((pv - running) / running).min() * 100
    win_rate   = (ret > 0).mean() * 100

    return {
        "label":         label,
        "total_ret":     round(total_ret, 2),
        "ann_ret":       round(ann_ret, 2),
        "ann_vol":       round(ann_vol, 2),
        "sharpe":        round(sharpe, 3),
        "max_dd":        round(max_dd, 2),
        "win_rate":      round(win_rate, 1),
        "final":         round(pv[-1], 2),
    }


# ══════════════════════════════════════════════════
# 7. EXECUÇÃO PRINCIPAL
# ══════════════════════════════════════════════════

import os

BASE = "/content"   # <- única mudança necessária

assets = {
    "BRFS3": os.path.join(BASE, "BRFS3_Dados_Históricos.csv"),
    "PETR4": os.path.join(BASE, "PETR4_Dados_Históricos.csv"),
    "VALE3": os.path.join(BASE, "VALE3_Dados_Históricos.csv"),
}

all_metrics = []

print("=" * 58)
print("  QuantumFinance – RL Agent (Dados Reais – Investing.com)")
print("=" * 58)

for name, path in assets.items():
    print(f"\n{'─'*58}")
    print(f"  ATIVO: {name}")
    print(f"{'─'*58}")

    df = load_investing_csv(path, name)
    df = add_indicators(df)
    n_rows = len(df)
    print(f"  Período: {df['Data'].min().date()} → {df['Data'].max().date()} ({n_rows} pregões)")

    agent, df_train, df_test = train(df, n_episodes=300, verbose=True, name=name)
    print(f"  Treino: {len(df_train)} pregões | Teste: {len(df_test)} pregões")

    _, rl_portfolio = evaluate(agent, df_test)
    bh_portfolio    = buy_and_hold(df_test)

    size = min(len(rl_portfolio), len(bh_portfolio))
    rl_portfolio = rl_portfolio[:size]
    bh_portfolio = bh_portfolio[:size]

    m_rl = metrics(rl_portfolio, f"{name} – RL Agent")
    m_bh = metrics(bh_portfolio, f"{name} – Buy & Hold")
    all_metrics += [m_rl, m_bh]

    for m in [m_rl, m_bh]:
        print(f"\n  {m['label']}")
        print(f"    Retorno Total     : {m['total_ret']:>8.2f}%")
        print(f"    Retorno Anualizado: {m['ann_ret']:>8.2f}%")
        print(f"    Volatilidade Anual: {m['ann_vol']:>8.2f}%")
        print(f"    Sharpe Ratio      : {m['sharpe']:>8.3f}")
        print(f"    Max Drawdown      : {m['max_dd']:>8.2f}%")
        print(f"    Win Rate (dias +) : {m['win_rate']:>8.1f}%")
        print(f"    Portfólio Final   : R${m['final']:>10,.2f}")

import pandas as pd
df_res = pd.DataFrame(all_metrics)[["label","total_ret","sharpe","max_dd","win_rate","final"]]
df_res.columns = ["Estratégia","Retorno%","Sharpe","MaxDD%","WinRate%","Final R$"]
print(df_res.to_string(index=False))
print("\n[OK] Simulação concluída.")


  QuantumFinance – RL Agent (Dados Reais – Investing.com)

──────────────────────────────────────────────────────────
  ATIVO: BRFS3
──────────────────────────────────────────────────────────
  Período: 1998-01-26 → 2025-07-23 (5000 pregões)
  [BRFS3] ep  75/300 | ε=0.687 | portfólio treino: R$31,590
  [BRFS3] ep 150/300 | ε=0.471 | portfólio treino: R$10,827
  [BRFS3] ep 225/300 | ε=0.324 | portfólio treino: R$22,941
  [BRFS3] ep 300/300 | ε=0.222 | portfólio treino: R$63,947
  Treino: 3500 pregões | Teste: 1500 pregões

  BRFS3 – RL Agent
    Retorno Total     :   -13.29%
    Retorno Anualizado:    -2.42%
    Volatilidade Anual:    23.33%
    Sharpe Ratio      :    0.011
    Max Drawdown      :   -51.90%
    Win Rate (dias +) :     11.9%
    Portfólio Final   : R$  8,670.71

  BRFS3 – Buy & Hold
    Retorno Total     :   -42.34%
    Retorno Anualizado:    -9.01%
    Volatilidade Anual:    50.26%
    Sharpe Ratio      :    0.063
    Max Drawdown      :   -86.67%
    Win Rate (dias +) 